In [243]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree,export_text
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
from sklearn.decomposition import PCA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import roc_curve, roc_auc_score

OPIS SKUPA PODATAKA

The dataset consists of 10 numerical and 8 categorical attributes.
The 'Revenue' attribute can be used as the class label.

"Administrative", "Administrative Duration", "Informational", "Informational Duration", "Product Related" and "Product Related Duration" represent the number of different types of pages visited by the visitor in that session and total time spent in each of these page categories. The values of these features are derived from the URL information of the pages visited by the user and updated in real time when a user takes an action, e.g. moving from one page to another. The "Bounce Rate", "Exit Rate" and "Page Value" features represent the metrics measured by "Google Analytics" for each page in the e-commerce site. The value of "Bounce Rate" feature for a web page refers to the percentage of visitors who enter the site from that page and then leave ("bounce") without triggering any other requests to the analytics server during that session. The value of "Exit Rate" feature for a specific web page is calculated as for all pageviews to the page, the percentage that were the last in the session. The "Page Value" feature represents the average value for a web page that a user visited before completing an e-commerce transaction. The "Special Day" feature indicates the closeness of the site visiting time to a specific special day (e.g. Mother’s Day, Valentine's Day) in which the sessions are more likely to be finalized with transaction. The value of this attribute is determined by considering the dynamics of e-commerce such as the duration between the order date and delivery date. For example, for Valentina’s day, this value takes a nonzero value between February 2 and February 12, zero before and after this date unless it is close to another special day, and its maximum value of 1 on February 8. The dataset also includes operating system, browser, region, traffic type, visitor type as returning or new visitor, a Boolean value indicating whether the date of the visit is weekend, and month of the year.

In [244]:
exel = []

In [245]:
def metrika(title,preds,y_test):
    print(title, accuracy_score(y_test,preds))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print("\nClassification Report:")
    print(classification_report(y_test, preds))
    print("-------------------------------------------------------------------")

In [246]:
def plot_roc_curve(fpr, tpr, roc_auc, model_name, set_name):

    output_image = 'slike/'+model_name+'+'+ set_name+'.png'
    plt.figure(figsize=(8,6))
    plt.plot(fpr, tpr, color='blue', label=f'{model_name} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guessing (AUC = 0.5)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig(output_image, dpi=300)
    plt.close()

In [247]:
def metrika1(model_name,y_pred,y_true,fpr,tpr,roc_auc):
    plot_roc_curve(fpr,tpr,roc_auc,model_name,'PCA')
    return {
        "model": model_name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, average="binary", zero_division=0)),
        "f1_score_false": float(f1_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "f1_score_true": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)), 
    }

In [248]:
def predicting(model, title, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    # importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    # print (importances)
    met = metrika1(title ,preds ,y_test,fpr,tpr,roc_auc)
    exel.append(met)
    # print(met)

In [249]:
df = pd.read_csv(r'podaci\online+shoppers+purchasing+intention+dataset\online_shoppers_intention preprocessed.csv', encoding='cp1252', sep=',')
print(df.columns)

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType',
       'Weekend', 'Revenue'],
      dtype='object')


In [250]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,1,1,1,1,2,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,2,2,2,1,2,2,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,4,1,9,3,2,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,2,3,2,2,4,2,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,2,3,3,1,4,2,1,0


In [251]:
X = df.copy(deep=True)

In [252]:
X = X.drop("Revenue", axis=1)
y = df["Revenue"]

In [253]:
best5 = ["PageValues","ProductRelated_Duration","BounceRates","ExitRates","ProductRelated"]
best10 = ["PageValues","ProductRelated_Duration","BounceRates","ExitRates","ProductRelated","Administrative_Duration","Month","Administrative","Region","TrafficType","Informational_Duration"]
feature_names = X.columns

In [254]:
numeric_features = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues"
]

In [255]:
pca = PCA(n_components=2)
scaler_minmax = MinMaxScaler()
scaler_standard = StandardScaler()

In [256]:
X_log = X.copy(deep=True)

In [257]:
X_log[numeric_features] = np.log1p(X_log[numeric_features])     # Logaritamsko skaliranje
X_sca_std = scaler_standard.fit_transform(X)                    # Standard scaler
X_sca_mm = scaler_minmax.fit_transform(X)                       # MinMax scaler
X_pca = pca.fit_transform(X)                                    # Samo PCA

X_pca_sca_std = pca.fit_transform(X_sca_std)                    # Standard scaler + PCA
X_pca_sca_mm = pca.fit_transform(X_sca_mm)                      # MinMax scaler + PCA

X_best5 =scaler_minmax.fit_transform(X[best5])
X_best10 = scaler_minmax.fit_transform(X[best10])


In [258]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123, stratify=y)

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_log, y, test_size=0.2, random_state=123, stratify=y)
X_train_sca_std, X_test_sca_std, y_train_sca_std, y_test_sca_std = train_test_split(X_sca_std, y, test_size=0.2, random_state=123, stratify=y)
X_train_sca_mm, X_test_sca_mm, y_train_sca_mm, y_test_sca_mm = train_test_split(X_sca_mm, y, test_size=0.2, random_state=123, stratify=y)
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2, random_state=123, stratify=y)

X_train_pca_sca_std, X_test_pca_sca_std, y_train_pca_sca_std, y_test_pca_sca_std = train_test_split(X_pca_sca_std, y, test_size=0.2, random_state=123, stratify=y)
X_train_pca_sca_mm, X_test_pca_sca_mm, y_train_pca_sca_mm, y_test_pca_sca_mm = train_test_split(X_pca_sca_mm, y, test_size=0.2, random_state=123, stratify=y)

X_train_best5, X_test_best5, y_train_best5, y_test_best5 = train_test_split(X_best5, y, test_size=0.2, random_state=123, stratify=y)
X_train_best10, X_test_best10, y_train_best10, y_test_best10 = train_test_split(X_best10, y, test_size=0.2, random_state=123, stratify=y)


In [259]:
datasets = [
    # ("Originalni podaci", X_train, X_test, y_train, y_test),
    # ("BEST5 podaci", X_train_best5, X_test_best5, y_train_best5, y_test_best5),
    # ("BEST10 podaci", X_train_best10, X_test_best10, y_train_best10, y_test_best10),
    
    # ("Log transformacija", X_train_log, X_test_log, y_train_log, y_test_log),
    # ("StandardScaler", X_train_sca_std, X_test_sca_std, y_train_sca_std, y_test_sca_std),
    # ("MinMaxScaler", X_train_sca_mm, X_test_sca_mm, y_train_sca_mm, y_test_sca_mm),
    ("PCA", X_train_pca, X_test_pca, y_train_pca, y_test_pca),

    # ("StandardScaler + PCA", X_train_pca_sca_std, X_test_pca_sca_std, y_train_pca_sca_std, y_test_pca_sca_std),
    # (" MinMaxScaler + PCA", X_train_pca_sca_mm, X_test_pca_sca_mm, y_train_pca_sca_mm, y_test_pca_sca_mm),

    # ("Engineering PCA", X_train_eng, X_test_eng, y_train_eng, y_test_eng),
    # ("Technical PCA", X_train_tec, X_test_tec, y_train_tec, y_test_tec),
    # ("Time PCA", X_train_time, X_test_time, y_train_time, y_test_time)
]

PREDVIDJANJE

In [260]:
dt = DecisionTreeClassifier()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(dt, f"Stablo odlucivanja + {name}", X_tr, y_tr, X_te, y_te)

In [261]:
rf = RandomForestClassifier()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(rf, f"Random Forest + {name}", X_tr, y_tr, X_te, y_te)

In [262]:
mlp = MLPClassifier(hidden_layer_sizes=(10,), max_iter=1000)

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(mlp, f"MLP + {name}", X_tr, y_tr, X_te, y_te)

In [263]:
nb = GaussianNB()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(nb, f"NB + {name}", X_tr, y_tr, X_te, y_te)

In [264]:
knn = KNeighborsClassifier(n_neighbors=3)

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(knn, f"KNN + {name}", X_tr, y_tr, X_te, y_te)

In [265]:
lr = LogisticRegression()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(lr, f"lr + {name}", X_tr, y_tr, X_te, y_te)

In [266]:
svm = SVC(kernel='rbf', probability=True)

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(svm, f"SVM + {name}", X_tr, y_tr, X_te, y_te)

In [267]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(xgb, f"XGB + {name}", X_tr, y_tr, X_te, y_te)

c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:42:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [268]:
# for name, X_tr, X_te, y_tr, y_te in datasets:

#     y_train_oh = to_categorical(y_tr)
#     y_test_oh = to_categorical(y_te)

#     model = Sequential([
#         Dense(128, input_shape=(X_tr.shape[1],), activation='relu'),
#         Dropout(0.3),
#         Dense(64, activation='relu'),
#         Dropout(0.3),
#         Dense(32, activation='relu'),
#         # Dropout(0.3),
#         Dense(16, activation='relu'),
#         # Dropout(0.3),
#         Dense(8, activation='relu'),
#         Dense(2, activation='softmax')
#     ])

#     model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

#     model.fit(X_tr, y_train_oh, epochs=50, batch_size=8, verbose=0)

#     y_pred_prob = model.predict(X_te)
#     y_pred = y_pred_prob.argmax(axis=1)

#     print(metrika1(name,y_pred , y_te))

In [269]:
df_exel = pd.DataFrame(exel)
df_exel.to_excel("rezKlasifikacije/rezultati_PCA.xlsx", index=False)